# Comparación de Arquitecturas: YOLOv8-seg vs. Mask R-CNN

Este notebook complementa `00_TP_Informe_Principal.ipynb`, agregando una
comparación de **arquitecturas** (no solo hiperparámetros) tal como sugirió
la cátedra: entrenamos **Mask R-CNN** (ResNet-50 + FPN, vía `torchvision`)
sobre el mismo dataset TACO y el mismo split train/val ya usado con
YOLOv8-seg, para comparar mAP, precision/recall y tiempo de inferencia
entre ambas arquitecturas.

Se entrenan además las 3 estrategias de transfer learning discutidas en la
materia, sobre esta arquitectura:
1. **Desde cero** (pesos aleatorios)
2. **Fine-tuning completo** (pesos pre-entrenados en COCO, todo entrenable)
3. **Feature extraction** (pesos pre-entrenados, backbone congelado)

**Importante:** este notebook asume que ya corriste
`00_TP_Informe_Principal.ipynb` de punta a punta al menos una vez (para que
exista `data/taco_yolo/images/{train,val}/`, que se usa acá para respetar
el mismo split).


## 0. Instalación de dependencias adicionales

In [ ]:
!pip install -q torchmetrics


## 1. Imports y configuración

Detectamos el dispositivo igual que en el notebook principal. Ojo: algunas
operaciones de Mask R-CNN (roi_align, nms) pueden no estar soportadas en
MPS en tu versión de PyTorch. Si alguna celda de entrenamiento tira un
error mencionando una operación no implementada en MPS, cambiá
`FORZAR_CPU = True` más abajo y volvé a correr desde esa celda.


In [ ]:
import json
import sys
import time
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torchvision
from PIL import Image, ImageDraw
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision

sys.path.append("../data")
from prepare_dataset import build_category_mapping

MACRO_CLASSES = ["plastico", "papel_carton", "vidrio", "metal", "organico", "otros"]
NUM_CLASSES = len(MACRO_CLASSES) + 1  # +1 por el fondo (clase 0 obligatoria en Mask R-CNN)

TACO_DIR = Path("../data/TACO/data")
ANNOTATIONS_PATH = TACO_DIR / "annotations.json"
TACO_YOLO_DIR = Path("../data/taco_yolo")

FORZAR_CPU = False
device = torch.device("cpu") if FORZAR_CPU else torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Dispositivo:", device)


## 2. Dataset

Lee las anotaciones COCO originales de TACO, pero respeta el **mismo
split** train/val que ya generó `prepare_dataset.py` para YOLOv8-seg
(determinado por qué imágenes quedaron copiadas en
`taco_yolo/images/{split}/`), para que la comparación entre arquitecturas
sea sobre exactamente los mismos conjuntos de datos.


In [ ]:
# ===== SOLO DEFINE LA CLASE, no carga nada todavía =====
class TacoMaskRCNNDataset(Dataset):
    def __init__(self, coco, split, cat_id_to_macro):
        self.cat_id_to_macro = cat_id_to_macro
        self.macro_to_class_id = {name: i + 1 for i, name in enumerate(MACRO_CLASSES)}

        images_by_id = {img["id"]: img for img in coco["images"]}
        anns_by_image = defaultdict(list)
        for ann in coco["annotations"]:
            anns_by_image[ann["image_id"]].append(ann)

        split_dir = TACO_YOLO_DIR / "images" / split
        valid_ids = set()
        for f in split_dir.glob("*"):
            id_str = f.name.split("_", 1)[0]
            if id_str.isdigit():
                valid_ids.add(int(id_str))

        self.samples = [
            (image_id, images_by_id[image_id], anns_by_image[image_id])
            for image_id in valid_ids
            if image_id in images_by_id and image_id in anns_by_image
        ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_id, img_info, anns = self.samples[idx]
        img = Image.open(TACO_DIR / img_info["file_name"]).convert("RGB")
        w, h = img.size

        boxes, labels, masks = [], [], []
        for ann in anns:
            macro = self.cat_id_to_macro.get(ann["category_id"], "otros")
            class_id = self.macro_to_class_id[macro]

            segmentation = ann.get("segmentation")
            if not segmentation or not isinstance(segmentation, list):
                continue

            mask_img = Image.new("L", (w, h), 0)
            draw = ImageDraw.Draw(mask_img)
            for poly in segmentation:
                if len(poly) < 6:
                    continue
                pts = [(poly[i], poly[i + 1]) for i in range(0, len(poly), 2)]
                draw.polygon(pts, fill=1)
            mask = np.array(mask_img)

            if mask.sum() == 0:
                continue
            ys, xs = np.where(mask)
            x1, x2, y1, y2 = xs.min(), xs.max(), ys.min(), ys.max()
            if x2 <= x1 or y2 <= y1:
                continue

            boxes.append([x1, y1, x2, y2])
            labels.append(class_id)
            masks.append(mask)

        if not boxes:  # imagen sin anotaciones válidas: caja dummy para no romper el batch
            boxes, labels, masks = [[0, 0, 1, 1]], [1], [np.zeros((h, w), dtype=np.uint8)]

        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "masks": torch.as_tensor(np.stack(masks), dtype=torch.uint8),
            "image_id": torch.tensor([image_id]),
        }
        img_tensor = torchvision.transforms.functional.to_tensor(img)
        return img_tensor, target


def collate_fn(batch):
    return tuple(zip(*batch))


In [ ]:
with open(ANNOTATIONS_PATH) as f:
    coco = json.load(f)

cat_id_to_macro = build_category_mapping(coco)

train_dataset = TacoMaskRCNNDataset(coco, "train", cat_id_to_macro)
val_dataset = TacoMaskRCNNDataset(coco, "val", cat_id_to_macro)
print(f"Train: {len(train_dataset)} imágenes | Val: {len(val_dataset)} imágenes")

BATCH_SIZE = 2
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)


## 3. Definición del modelo (parametrizado por estrategia)

`modo=0` desde cero, `modo=1` fine-tuning completo, `modo=2` feature
extraction (backbone congelado) — mismo criterio usado en la comparación
de AlexNet/ResNet vista en la materia.


In [ ]:
# ===== SOLO DEFINE LA FUNCIÓN, no ejecuta nada al correrla =====
def get_model(num_classes, modo):
    weights = None if modo == 0 else MaskRCNN_ResNet50_FPN_Weights.DEFAULT
    model = maskrcnn_resnet50_fpn(weights=weights)

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, num_classes)

    if modo == 2:  # feature extraction: congelamos backbone
        for p in model.backbone.parameters():
            p.requires_grad = False

    return model


## 4. Funciones de entrenamiento y evaluación

In [ ]:
# ===== ESTA CELDA SOLO DEFINE LA FUNCIÓN, NO ENTRENA NADA =====
# Correrla no tarda nada (es instantáneo) y no imprime nada — es normal.
# El entrenamiento real pasa en la CELDA SIGUIENTE, la que la LLAMA.

def entrenar_maskrcnn(modo, nombre, num_epochs=5):
    model = get_model(NUM_CLASSES, modo).to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

    print(f"\nEntrenando: {nombre} (modo={modo})")
    print(f"Total de iteraciones por época: {len(train_loader)}")
    historial = []
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        t0 = time.time()
        for i, (imgs, targets) in enumerate(train_loader):
            imgs = [img.to(device) for img in imgs]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(imgs, targets)
            losses = sum(loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()
            epoch_loss += losses.item()

            if (i + 1) % 20 == 0:
                elapsed = time.time() - t0
                print(f"  Época {epoch+1}, iteración {i+1}/{len(train_loader)} "
                      f"- loss actual: {losses.item():.4f} - {elapsed:.1f}s transcurridos")

        avg_loss = epoch_loss / len(train_loader)
        elapsed = time.time() - t0
        historial.append(avg_loss)
        print(f"Época {epoch+1}/{num_epochs} completa - loss promedio: {avg_loss:.4f} - tiempo: {elapsed:.1f}s")

    Path("../runs").mkdir(exist_ok=True)
    torch.save(model.state_dict(), f"../runs/maskrcnn_{nombre}.pt")
    return model, historial


## 5. Corrida 1: Desde cero

Entrenamiento sin pesos pre-entrenados. Se espera un desempeño bajo, dado
el tamaño reducido del dataset — sirve como punto de comparación para
cuantificar el aporte real del transfer learning.


In [ ]:
# ===== ESTA CELDA SÍ ENTRENA (tarda tiempo real) =====
# Empezamos con num_epochs=1 para medir cuánto tarda una época antes de
# comprometernos a las 5 completas. Cuando confirmemos el tiempo, subimos
# este número.

model_scratch, hist_scratch = entrenar_maskrcnn(modo=0, nombre="scratch", num_epochs=1)


In [ ]:
resultado_scratch = evaluar_maskrcnn(model_scratch, "Desde cero")


## 6. Corrida 2: Fine-tuning completo

In [ ]:
# ===== ESTA CELDA SÍ ENTRENA (tarda tiempo real) =====
# Ajustá num_epochs según el tiempo que midas en la Corrida 1.
model_ft, hist_ft = entrenar_maskrcnn(modo=1, nombre="finetuning", num_epochs=5)


In [ ]:
resultado_ft = evaluar_maskrcnn(model_ft, "Fine-tuning completo")


## 7. Corrida 3: Feature extraction

In [ ]:
# ===== ESTA CELDA SÍ ENTRENA (tarda tiempo real) =====
# Ajustá num_epochs según el tiempo que midas en la Corrida 1.
model_fe, hist_fe = entrenar_maskrcnn(modo=2, nombre="feature_extraction", num_epochs=5)


In [ ]:
resultado_fe = evaluar_maskrcnn(model_fe, "Feature extraction")


## 8. Tabla comparativa final

Comparamos las 3 estrategias de Mask R-CNN entre sí, y contra el mejor
resultado ya obtenido con YOLOv8-seg en el notebook principal
(mAP50 mask = 0.223, mAP50-95 mask = 0.156, ~5-10 ms/imagen en MPS).


In [ ]:
import pandas as pd

resultados = [resultado_scratch, resultado_ft, resultado_fe]
df = pd.DataFrame(resultados).set_index("nombre")

# Referencia: mejor resultado ya obtenido con YOLOv8-seg (notebook principal)
df.loc["YOLOv8s-seg (referencia)"] = {
    "mAP50_box": 0.251,
    "mAP50-95_box": 0.193,
    "mAP50_mask": 0.223,
    "mAP50-95_mask": 0.156,
    "ms_por_imagen": None,  # completar con el valor real impreso en la evaluación del notebook principal
}

df


## 9. Conclusiones de esta comparación

Completar tras correr las 3 estrategias, con base en los números de la
Sección 8. Preguntas guía:

- ¿Qué tan grande es la diferencia entre "desde cero" y las otras dos
  estrategias? ¿Confirma la necesidad de transfer learning en un dataset
  chico, como se planteó en el notebook principal?
- ¿Fine-tuning completo superó a feature extraction? ¿Es consistente con
  lo observado en la comparación de AlexNet/ResNet vista en la materia?
- ¿Cómo se compara el mejor resultado de Mask R-CNN contra YOLOv8-seg, en
  mAP y en tiempo de inferencia? ¿Qué arquitectura es más apropiada para
  el caso de uso (cinta transportadora en tiempo real) planteado en el
  proyecto?
